[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/prompts/react_prompting.ipynb)


<center>
<p style="text-align:center">
<img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
<br>
<br>
<a href="https://arize.com/docs/phoenix/">文档</a>
|
<a href="https://github.com/Arize-ai/phoenix">GitHub</a>
|
<a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
</p>
</center>
<h1 align="center">ReAct 提示教程</h1>

**ReAct（推理与行动）**是一种引导 LLM 在行动前逐步思考的提示技术。与模型直接给出答案的传统提示方式相比，ReAct 先引导模型分析问题，再决定需要哪些工具或行动来得到最佳方案。

ReAct 适合需要**借助外部工具分步解决问题**的场景。通过展示选择各个工具的推理依据，它还提升了**透明度**，便于理解和改进模型行为。

本教程将：
- 学习在 Phoenix 中设计提示词、工具和评估器。
- 逐步改进提示词，理解 ReAct 的作用。
- 借助 Phoenix 和 LLM 评审技术逐步评估准确性，观察模型推理过程。
- 学习在实际场景中使用 ReAct 改善任务执行和问题求解。

⚠️ 本教程需要 OpenAI API 密钥。

开始吧！🚀

# **配置依赖与密钥**

In [ ]:
%pip install -qqq "arize-phoenix-client" "arize-phoenix-otel" "arize-phoenix-evals>=0.28.1" datasets openinference-instrumentation-openai

接下来连接 Phoenix。

In [ ]:
import os
from getpass import getpass

if not os.environ.get("PHOENIX_COLLECTOR_ENDPOINT"):
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = input("请输入 Phoenix 采集端地址：")
if not os.environ.get("PHOENIX_API_KEY"):
    os.environ["PHOENIX_API_KEY"] = getpass("请输入 Phoenix API 密钥：")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("请输入 OpenAI API 密钥：")

In [ ]:
import nest_asyncio
from openai import OpenAI
from openai.types.chat.completion_create_params import CompletionCreateParamsBase
from phoenix.client import AsyncClient
from phoenix.evals import (
    LLM,
)
from phoenix.evals.metrics import ToolInvocationEvaluator, ToolSelectionEvaluator

px_client = AsyncClient()

In [ ]:
nest_asyncio.apply()

## **为应用接入埋点**

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="ReAct-Examples", auto_instrument=True)

# **将数据集加载到 Phoenix**

此数据集包含 20 个顾客可能向商店聊天机器人提出的客服问题。探索 ReAct 提示时，将利用这些问题引导 LLM 选择合适工具。

这里还导入 Phoenix 客户端，以便直接在 Notebook 中创建和修改提示词，并将变更同步到 Phoenix 界面。

运行本单元后，应能在 Phoenix 的数据集页面找到该数据集。

In [ ]:
import uuid

from datasets import load_dataset

ds = load_dataset("syeddula/customer_questions")["train"]
df = ds.to_pandas()

unique_id = uuid.uuid4()

# 将数据集上传到 Phoenix
dataset = await px_client.datasets.create_dataset(
    name=f"customer-questions-{unique_id}",
    dataframe=df,
    input_keys=["Questions"],
)

# **定义工具**

接下来定义 LLM 可使用的工具。这里有五种各司其职的工具：商品比较、商品详情、折扣、客户支持和包裹追踪。

LLM 会根据顾客的问题确定最佳工具调用顺序。


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "product_comparison",
            "description": "比较两件商品的特性。",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_a_id": {
                        "type": "string",
                        "description": "商品 A 的唯一标识符。",
                    },
                    "product_b_id": {
                        "type": "string",
                        "description": "商品 B 的唯一标识符。",
                    },
                },
                "required": ["product_a_id", "product_b_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "product_details",
            "description": "获取某件商品的详细特性。",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_id": {
                        "type": "string",
                        "description": "商品的唯一标识符。",
                    }
                },
                "required": ["product_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_discount_code",
            "description": "查询折扣和促销活动。 为订单应用折扣码。",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "integer",
                        "description": "订单的唯一标识符。",
                    },
                    "discount_code": {
                        "type": "string",
                        "description": "要应用的折扣码。",
                    },
                },
                "required": ["order_id", "discount_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "customer_support",
            "description": "获取处理相关问题的客服联系方式。",
            "parameters": {
                "type": "object",
                "properties": {
                    "issue_type": {
                        "type": "string",
                        "description": "问题类型（例如账单、技术支持）。",
                    }
                },
                "required": ["issue_type"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "track_package",
            "description": "根据物流单号跟踪包裹状态。",
            "parameters": {
                "type": "object",
                "properties": {
                    "tracking_number": {
                        "type": "integer",
                        "description": "包裹的物流单号。",
                    }
                },
                "required": ["tracking_number"],
            },
        },
    },
]

# **初始提示词**

先定义一个简单提示词，要求系统使用可用工具回答问题。具体选择哪些工具、如何使用，交由模型根据顾客每次提问的上下文决定。

In [ ]:
from phoenix.client.types import PromptVersion

params = CompletionCreateParamsBase(
    model="gpt-4",
    tools=tools,
    tool_choice="auto",
    messages=[
        {
            "role": "system",
            "content": """你是一名乐于助人的客服 Agent，任务是确定最适合回答顾客问题的工具。
            """,
        },
        {"role": "user", "content": "{{questions}}"},
    ],
)

prompt_identifier = "customer-support-prompt"

prompt = await px_client.prompts.create(
    name=prompt_identifier,
    prompt_description="客户支持",
    version=PromptVersion.from_openai(params),
)

此时，初始提示词已出现在 Phoenix 的提示词页面。后续任何修改都会记录在**版本（Versions）**中，便于跟踪和比较变化。

Phoenix 存储的不只有文本，还包括提示词模板、模型配置和响应格式等关键信息，让输出生成过程保持结构化和一致性。

![初始提示词](https://storage.googleapis.com/arize-phoenix-assets/assets/images/react_prompt1.png)

接下来定义工具调用评测的提示词模板。本步骤使用 **[LLM 评审](https://arize.com/docs/phoenix/evaluation/concepts-evals/llm-as-a-judge)**，即让一个 LLM 评估另一个 LLM 的表现。

这条提示词交给评审模型，同时输入用户问题和系统选择的工具。模型通过推理判断所选工具能否有效解决问题，并解释评估结果。

In [ ]:
AVAILABLE_TOOLS = """
product_comparison: 比较两件商品的特性。
product_details: 获取某件商品的详细特性。
apply_discount_code: 为订单应用折扣码。
customer_support: 获取处理相关问题的客服联系方式。
track_package: 根据物流单号跟踪包裹状态。
"""

接下来的单元先定义实验任务，再在 `evaluate_tool_selection` 和 `evaluate_tool_invocation` 函数中，使用 Phoenix 内置的 `ToolSelectionEvaluator` 与 `ToolInvocationEvaluator` 定义 LLM 评估器，最后运行实验。

In [ ]:
eval_llm = LLM(provider="openai", model="gpt-4o-mini")


def prompt_task(input):
    client = OpenAI()
    resp = client.chat.completions.create(
        **prompt.format(variables={"questions": input["Questions"]})
    )
    return resp


def evaluate_tool_selection(input, output):
    evaluator = ToolSelectionEvaluator(llm=eval_llm)
    result = evaluator.evaluate(
        eval_input={
            "input": input["Questions"],
            "available_tools": AVAILABLE_TOOLS,
            "tool_selection": str(output),
        }
    )
    return result[0].score


def evaluate_tool_invocation(input, output):
    evaluator = ToolInvocationEvaluator(llm=eval_llm)
    result = evaluator.evaluate(
        eval_input={
            "input": input["Questions"],
            "available_tools": AVAILABLE_TOOLS,
            "tool_selection": str(output),
        }
    )
    return result[0].score

### **实验**

In [ ]:
initial_experiment = await px_client.experiments.run_experiment(
    dataset=dataset,
    task=prompt_task,
    evaluators=[evaluate_tool_selection, evaluate_tool_invocation],
    experiment_description="客服提示词",
    experiment_name="initial-prompt",
    experiment_metadata={"prompt_id": prompt.id},
)

运行实验与评测后，可以进一步分析结果。点击实验可查看 LLM 为具体输入选择的工具；再点击评测追踪，可以查看 LLM 评审对输出打分的依据。

![第一次运行](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/react_prompt.gif)

# **ReAct 提示词**

接下来使用 **ReAct 提示**技术改进系统提示词：强调模型应逐步分析问题、按逻辑拆解，再决定使用哪些工具以及调用顺序，并输出对应工具及其参数。

与初始提示词相比，这种方式强调先推理再行动，引导模型根据问题的具体上下文选择最佳工具和参数，而非直接套用预设行动。

In [ ]:
params = CompletionCreateParamsBase(
    model="gpt-4",
    tools=tools,
    tool_choice="auto",
    messages=[
        {
            "role": "system",
            "content": """ 你是一名乐于助人的客服 Agent。仔细分析顾客的问题，充分理解其需求。
            第一步：逐步思考，确定回答问题所需的关键信息，并考虑这些信息之间的依赖关系。
            第二步：决定使用哪些工具。如需多个工具，确定正确的调用顺序。
            第三步：输出所选工具及相关参数。
            """,
        },
        {"role": "user", "content": "{{questions}}"},
    ],
)

prompt_identifier = "customer-support-prompt"

prompt = await px_client.prompts.create(
    name=prompt_identifier,
    prompt_description="客服 ReAct 提示词",
    version=PromptVersion.from_openai(params),
)

在提示词页面可以看到更新后的内容。持续迭代会形成版本历史。

![第二版提示词](https://storage.googleapis.com/arize-phoenix-assets/assets/images/react_prompt2.png)

与前面一样，定义任务、构建评估器并运行实验。

In [ ]:
def prompt_task(input):
    client = OpenAI()
    resp = client.chat.completions.create(
        **prompt.format(variables={"questions": input["Questions"]})
    )
    return resp

### **实验**

In [ ]:
initial_experiment = await px_client.experiments.run_experiment(
    dataset=dataset,
    task=prompt_task,
    evaluators=[evaluate_tool_selection, evaluate_tool_invocation],
    experiment_description="客服提示词",
    experiment_name="improved-prompt",
    experiment_metadata={"prompt_id": prompt.id},
)

使用更新后的 ReAct 提示词后，**LLM 评估器**将更多输出判定为正确。点击追踪可以了解改进的原因。引导模型更仔细、有目的地处理问题，有助于观察 ReAct 的推理和行动过程。

可以查看评估器输出，进一步理解具体改进。

请注意，随机性和模型的非确定性行为会导致结果波动。


![评测结果](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/react_prompts2.gif)

要继续完善这些提示词并在其他数据集上测试，可尝试思维链（CoT）等方法，判断它们在具体场景中与 ReAct 的互补之处或差异。Phoenix 的 TypeScript 和 Python 客户端可将此过程接入工作流。

接下来可以查看更多 [Phoenix 示例](https://arize.com/docs/phoenix/notebooks)。如果还没有为项目加星，也欢迎[在 GitHub 上支持我们](https://github.com/Arize-ai/phoenix)！⭐️
